# Conditional Random Fields — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Normalize labels conditionally on the observed sequence
Linear-chain CRFs score entire label sequences with feature weights, then normalize over labels for the given input. These examples enumerate sequence scores, compute the partition function, marginals, Viterbi path, and transition-weight effects.

In [ ]:
# 1) enumerate CRF scores for length-3 binary labels
x=np.array([1.0,-0.5,1.2]); w=1.0; trans=0.7; seqs=list(itertools.product([0,1],repeat=3))
def score(y): return sum(w*x[t]*(1 if y[t]==1 else -1) for t in range(3))+sum(trans*(y[t]==y[t-1]) for t in range(1,3))
scores=np.array([score(y) for y in seqs]); probs=softmax(scores)
plt.figure(figsize=(6,3)); plt.bar(range(8),probs); plt.title('conditional distribution over label sequences')
assert abs(probs.sum()-1)<1e-12

In [ ]:
# 2) partition function Z(x) sums only over labels
Z=np.exp(scores).sum(); plt.figure(figsize=(6,3)); plt.bar(['Z(x)'],[Z]); plt.title(f'partition = {Z:.3f}')
assert abs(Z-47.15101323268092)<1e-9

In [ ]:
# 3) marginal P(y_t=1 | x) from sequence probabilities
marg=np.array([sum(p for p,y in zip(probs,seqs) if y[t]==1) for t in range(3)])
plt.figure(figsize=(6,3)); plt.plot(marg,'-o'); plt.ylim(0,1); plt.title('label marginals')
assert marg[0]>0.7 and marg[1]<0.7

In [ ]:
# 4) Viterbi sequence is the highest-scoring label path
best=seqs[int(np.argmax(scores))]; plt.figure(figsize=(6,3)); plt.step(range(3),best,where='mid'); plt.ylim(-.2,1.2); plt.title(f'Viterbi path {best}')
assert best==(1,1,1)

In [ ]:
# 5) stronger transition weight smooths labels
vals=[]
for tr in [0,0.7,2.0]:
    trans=tr; sc=np.array([score(y) for y in seqs]); pr=softmax(sc); vals.append(sum(pr[i] for i,y in enumerate(seqs) if y[0]==y[1]==y[2]))
plt.figure(figsize=(6,3)); plt.plot([0,0.7,2.0],vals,'-o'); plt.xlabel('transition weight'); plt.ylabel('P(all labels equal)'); plt.title('transition features encourage smoothness')
assert vals[-1]>vals[0]